## Feature Score

Goal: Combining all feature selections from the different models as well as their scores. 
1. load all df
2. normalize each
3. add the scores
4. normalize results
   

In [2]:
#import needed packages
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np

In [3]:
#Read in Data and check 
PM = pd.read_csv("data/featureselection_PM_score.csv")
CV = pd.read_csv("data/featureselection_cramersV_LGBMscore.csv")
gini = pd.read_csv("data/featureselection_gini_score.csv")

print(f" permutation: \n {PM.head()}")
print(f" gini: \n {gini.head()}")
print(f" Cramers V: \n {CV.head()}")

 permutation: 
    Unnamed: 0              base_name  imp_grouped
0          13          obamaapp_2016     0.016667
1           7  healthreformbill_2016     0.011285
2          17      pp_primary16_2016     0.003993
3          14              pid3_2016     0.002951
4          11       immi_muslim_2016     0.002431
 gini: 
    Unnamed: 0              base_name  imp_grouped
0           8  healthreformbill_2016     0.211796
1          13          obamaapp_2016     0.153091
2          21     univhealthcov_2016     0.124418
3          16    policies_favor_2016     0.078806
4           7    gvmt_involment_2016     0.047368
 Cramers V: 
    Unnamed: 0  n_features          added_feature  auc_lr_train  auc_lr_test  \
0           0           1          obamaapp_2016        0.8838       0.8677   
1           1           2  healthreformbill_2016        0.9379       0.9077   
2           2           3      pp_primary16_2016        0.9519       0.9292   
3           3           4              pid3_2

In [4]:
# normalize needed columns with MinMaxScaler()
# source: https://www.geeksforgeeks.org/python/normalize-a-column-in-pandas/

PM["norm_score_PM"] = MinMaxScaler().fit_transform(np.array(PM["imp_grouped"]).reshape(-1,1))
gini["norm_score_gini"] = MinMaxScaler().fit_transform(np.array(gini["imp_grouped"]).reshape(-1,1))
CV["norm_score_CV"] = MinMaxScaler().fit_transform(np.array(CV["auc_lr_test"]).reshape(-1,1)) #could also use: auc_lgbm_test



In [5]:
#check df
print(f" permutation: \n {PM.head()}")
print(f" gini: \n {gini.head()}")
print(f" Cramers V: \n {CV.head()}")

 permutation: 
    Unnamed: 0              base_name  imp_grouped  norm_score_PM
0          13          obamaapp_2016     0.016667       1.000000
1           7  healthreformbill_2016     0.011285       0.712963
2          17      pp_primary16_2016     0.003993       0.324074
3          14              pid3_2016     0.002951       0.268519
4          11       immi_muslim_2016     0.002431       0.240741
 gini: 
    Unnamed: 0              base_name  imp_grouped  norm_score_gini
0           8  healthreformbill_2016     0.211796         1.000000
1          13          obamaapp_2016     0.153091         0.720594
2          21     univhealthcov_2016     0.124418         0.584131
3          16    policies_favor_2016     0.078806         0.367039
4           7    gvmt_involment_2016     0.047368         0.217414
 Cramers V: 
    Unnamed: 0  n_features          added_feature  auc_lr_train  auc_lr_test  \
0           0           1          obamaapp_2016        0.8838       0.8677   
1          

In [6]:
#rename feature names to match, so .merge works 
PM_ = PM.rename(columns={"base_name": "feature"})
gini_ = gini.rename(columns={"base_name": "feature"})
CV_ = CV.rename(columns={"added_feature": "feature"})

In [7]:
#merge df and add scores together
# with the help of ChatGPT
# Prompts: 
"""
#PM["combined_scores"] = CV["norm_score"] + PM["norm_score"] + gini["norm_score"]
This does not work as the columns don't match by index. So I asked ChatGPT to fix it using this: 

I want to add some scores of PDS: 
PM["combined_scores"] = CV["norm_score"] + PM["norm_score"] + gini["norm_score"] //
BUT not all my rows match, and there are some rows that exist in one df but not in the other. 
Can you fix it and create a new df named FSC (feature selection score

i have 3 df with similar features.
the features are ordered differently and have slightly different names. 
I want to create a new df that combines the values for each feature into a new df 
-> i.e. df one as Apple: 1, Pear: 0.5, Kiwi: 0.4 CV has Pear: 0.7, Cherry: 1, Apple: 0.4, Orange: 0.8 
and it should return: combined_score which should look like this: 
Apple: 1.4, Kiwi: 0.4, Pear: 1.3, Cherry: 1, Orange: 0.8
"""

'\n#PM["combined_scores"] = CV["norm_score"] + PM["norm_score"] + gini["norm_score"]\nThis does not work as the columns don\'t match by index. So I asked ChatGPT to fix it using this: \n\nI want to add some scores of PDS: \nPM["combined_scores"] = CV["norm_score"] + PM["norm_score"] + gini["norm_score"] //\nBUT not all my rows match, and there are some rows that exist in one df but not in the other. \nCan you fix it and create a new df named FSC (feature selection score\n\ni have 3 df with similar features.\nthe features are ordered differently and have slightly different names. \nI want to create a new df that combines the values for each feature into a new df \n-> i.e. df one as Apple: 1, Pear: 0.5, Kiwi: 0.4 CV has Pear: 0.7, Cherry: 1, Apple: 0.4, Orange: 0.8 \nand it should return: combined_score which should look like this: \nApple: 1.4, Kiwi: 0.4, Pear: 1.3, Cherry: 1, Orange: 0.8\n'

In [8]:
#merge df and add scores together
# with the help of ChatGPT
"""
#cannot combine df on feature name when the following code is applied: 
PM_ = PM[["feature", "norm_score_PM"]]
gini_ = gini[["feature", "norm_score_gini"]]
CV_ = CV[["feature", "norm_score_CV"]]
"""
# --- Step 3: Merge all features (outer join!) ---
FSC = PM_.merge(gini_, on="feature", how="outer")
FSC = FSC.merge(CV_, on="feature", how="outer")

# --- Step 4: Replace missing values with 0 ---
FSC = FSC.fillna(0)

# --- Step 5: Create combined score ---
FSC["combined_score"] = (
    FSC["norm_score_PM"] +
    FSC["norm_score_gini"] +
    FSC["norm_score_CV"]
)

# --- Step 6: Sort features by importance ---
FSC = FSC.sort_values("combined_score").reset_index(drop=True)

# Optional: keep only relevant columns
FSC = FSC[["feature", "combined_score"]]
print(FSC)


                        feature  combined_score
0            race_overcome_2016        0.000000
1   reverse_discrimination_2016        0.000836
2               race_slave_2016        0.002201
3                    ideo5_2016        0.017742
4                        ft_fem        0.030321
5                        ft_blm        0.173083
6             pp_primary16_2016        1.061322
7                  imiss_d_2016        1.070336
8                  taxdoug_2016        1.081672
9          immi_naturalize_2016        1.092641
10        view_transgender_2016        1.094452
11         RIGGED_SYSTEM_4_2016        1.096720
12         gender_equality_2016        1.097039
13         RIGGED_SYSTEM_3_2016        1.102140
14           affirmact_gen_2016        1.118710
15             immi_muslim_2016        1.156171
16                 imiss_l_2016        1.161189
17       immi_contribution_2016        1.210079
18           police_threat_2016        1.243928
19                    pid3_2016        1

In [9]:
# normalize and save final feature selection and scores
FSC["norm_score_c"] = MinMaxScaler().fit_transform(np.array(FSC["combined_score"]).reshape(-1,1))
FSC.to_csv("data/featureselection_combined_scores.csv")

## further filerting: 
1. only keep top 20
2. remove variables that did not improve the test accuracy as viewd in the plots 

In [10]:
# filter for top 20
df = FSC.iloc[8:28]
print(df)
#len(df)

                   feature  combined_score  norm_score_c
8             taxdoug_2016        1.081672      0.523872
9     immi_naturalize_2016        1.092641      0.529184
10   view_transgender_2016        1.094452      0.530061
11    RIGGED_SYSTEM_4_2016        1.096720      0.531160
12    gender_equality_2016        1.097039      0.531314
13    RIGGED_SYSTEM_3_2016        1.102140      0.533784
14      affirmact_gen_2016        1.118710      0.541809
15        immi_muslim_2016        1.156171      0.559953
16            imiss_l_2016        1.161189      0.562383
17  immi_contribution_2016        1.210079      0.586061
18      police_threat_2016        1.243928      0.602455
19               pid3_2016        1.249009      0.604916
20     gvmt_involment_2016        1.259312      0.609906
21           govt_reg_2016        1.262574      0.611485
22            envwarm_2016        1.278629      0.619261
23             wealth_2016        1.296262      0.627801
24     policies_favor_2016     

In [11]:
#remove features based on accuracy scores in the individual nopebooks for CV and RF
ft_toremove = ["immi_naturalize_2016", "RIGGED_SYSTEM_3_2016", "immi_muslim_2016", "imiss_l_2016", "govt_reg_2016", "envwarm_2016", "univhealthcov_2016"]

In [12]:
df = df[~df["feature"].isin(ft_toremove)]
print(df)
len(df) #13
df.to_csv("data/featureselection_combined_scores_filtered.csv")

                   feature  combined_score  norm_score_c
8             taxdoug_2016        1.081672      0.523872
10   view_transgender_2016        1.094452      0.530061
11    RIGGED_SYSTEM_4_2016        1.096720      0.531160
12    gender_equality_2016        1.097039      0.531314
14      affirmact_gen_2016        1.118710      0.541809
17  immi_contribution_2016        1.210079      0.586061
18      police_threat_2016        1.243928      0.602455
19               pid3_2016        1.249009      0.604916
20     gvmt_involment_2016        1.259312      0.609906
23             wealth_2016        1.296262      0.627801
24     policies_favor_2016        1.450055      0.702285
26           obamaapp_2016        1.720594      0.833312
27   healthreformbill_2016        2.064766      1.000000


In [14]:
# save final 2012 data frame (mit allen werten auch - also das vom anfang, dann nochmal das durchs na data cleaning machen)
#make df with selected variables
fts = df['feature'].values.tolist() + ["presvote16post_2016"]
#print(fts)

df_total = pd.read_csv("data/VOTER_Survey_December16_Release1.csv")

df_selectedfeatures_2016 = df_total[fts]
df_selectedfeatures_2016.to_csv("data/selectedfeatures_2016_total.csv") #no NA :)
print(df_selectedfeatures_2016.head())

/tmp/ipykernel_61/3000705142.py:6: DtypeWarning: Columns (0: votereg_fnd_baseline, 1: religpew_muslim_baseline) have mixed types. Specify dtype option on import or set low_memory=False.
  df_total = pd.read_csv("data/VOTER_Survey_December16_Release1.csv")


  taxdoug_2016                              view_transgender_2016  \
0          Yes  Should be allow to us the restrooms of the gen...   
1           No  Should be required to use the restrooms of the...   
2          Yes  Should be allow to us the restrooms of the gen...   
3          Yes  Should be allow to us the restrooms of the gen...   
4           No  Should be required to use the restrooms of the...   

  RIGGED_SYSTEM_4_2016                    gender_equality_2016  \
0    Strongly disagree                              Don't know   
1       Strongly agree  Men have more opportunities than women   
2    Strongly disagree  Men have more opportunities than women   
3    Strongly disagree  Women have more opportunities than men   
4       Strongly agree  Women have more opportunities than men   

  affirmact_gen_2016      immi_contribution_2016         police_threat_2016  \
0              Favor  Mostly make a contribution                 Don't know   
1             Oppose          